## Welcome to Week 4, Day 4

This is the start of an AWESOME project! Really simple and very effective.

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
# from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
# from langchain.agents import Tool
from langchain_core.tools import Tool


from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
# load_dotenv(override=True)

In [ ]:
BASE_URL = "https://models.github.ai/inference"
MODEL_GPT_4o_MINI = "gpt-4o-mini"
MODEL_NAME = MODEL_GPT_4o_MINI
API_KEY = "github_pat_********"

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "lsv2_pt_*******"
os.environ["LANGSMITH_PROJECT"] = "myproject"

os.environ["SERPER_API_KEY"] = "4240****"

### Asynchronous LangGraph

To run a tool:  
Sync: `tool.run(inputs)`  
Async: `await tool.arun(inputs)`

To invoke the graph:  
Sync: `graph.invoke(state)`  
Async: `await graph.ainvoke(state)`

In [13]:
class State(TypedDict):
    
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_url = "https://api.pushover.net/1/messages.json"

# def push(text: str):
#     """Send a push notification to the user"""
#     requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

def push(message):
    print(f"🔥 [PUSH EMULATION]: {message}")

tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

## Extra installation step - if you don't have Node and Playwright on your computer

Next, you need to install NodeJS and Playwright on your computer if you don't already have them. Please see instructions here:

In [14]:
%pip install playwright -q
%pip list | grep -E "playwright"

!playwright install --with-deps -q
%pip install beautifulsoup4 -q

Note: you may need to restart the kernel to use updated packages.
playwright                  1.61.0
Note: you may need to restart the kernel to use updated packages.
error: unknown option '-q'
Note: you may need to restart the kernel to use updated packages.


In [15]:
# Introducing nest_asyncio
# Python async code only allows for one "event loop" processing aynchronous events.
# The `nest_asyncio` library patches this, and is used for special situations, if you need to run a nested event loop.

import nest_asyncio
nest_asyncio.apply()

### The LangChain community

One of the remarkable things about LangChain is the rich community around it.

Check this out:


In [21]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser

async_browser =  create_async_playwright_browser(headless=False)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [22]:
for tool in tools:
    print(f"{tool.name}={tool}")

click_element=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marley/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome> version=149.0.7827.55>
navigate_browser=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marley/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome> version=149.0.7827.55>
previous_webpage=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marley/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome> version=149.0.7827.55>
extract_text=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marley/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome> version=149.0.7827.55>
extract_hyperlinks=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marley/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome> version=149.0.7827.55>
get_elements=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/marle

In [24]:
tool_dict = {tool.name:tool for tool in tools}

navigate_tool = tool_dict.get("navigate_browser")
extract_text_tool = tool_dict.get("extract_text")

await navigate_tool.arun({"url": "https://www.cnn.com"})
text = await extract_text_tool.arun({})

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


TimeoutError: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://www.cnn.com/", waiting until "load"


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
import textwrap
print(textwrap.fill(text))

NameError: name 'text' is not defined

In [ ]:
all_tools = tools + [tool_push]

In [ ]:
llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=API_KEY
)
llm_with_tools = llm.bind_tools(all_tools)


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "10"}}

async def chat(user_input: str, history):
    result = await graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat).launch()